# 🫀 Cardiovascular Disease Risk Prediction & Clinical ML Pipeline
**Author:** Arjuna Fransesco  
**Domain:** Healthcare AI / Clinical Decision Support Systems  
**Dataset:** UCI Cleveland Heart Disease Dataset (303 patient records, 14 clinical features)

---

## 📌 1. Project Overview & Clinical Objective
Cardiovascular diseases (CVDs) are the leading cause of mortality globally. Early diagnostic stratification empowers healthcare professionals to intervene with preventative therapies before adverse cardiac events occur.

### Clinical Features Breakdown:
- `age`: Age of the patient in years
- `sex`: Biological sex (1 = male, 0 = female)
- `cp`: Chest pain type (0 = typical angina, 1 = atypical angina, 2 = non-anginal pain, 3 = asymptomatic)
- `trestbps`: Resting blood pressure (in mm Hg on admission to hospital)
- `chol`: Serum cholesterol in mg/dl
- `fbs`: Fasting blood sugar > 120 mg/dl (1 = true, 0 = false)
- `restecg`: Resting electrocardiographic results (0, 1, 2)
- `thalach`: Maximum heart rate achieved during exercise stress testing
- `exang`: Exercise-induced angina (1 = yes, 0 = no)
- `oldpeak`: ST depression induced by exercise relative to rest
- `slope`: The slope of the peak exercise ST segment (0, 1, 2)
- `ca`: Number of major vessels (0-3) colored by flourosopy
- `thal`: Thalassemia stress test result (1 = fixed defect, 2 = normal, 3 = reversible defect)
- `target`: Heart disease diagnosis (1 = present, 0 = absent)

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visual Styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11

# Load Dataset
df = pd.read_csv('../data/raw/heart.csv')
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 🔍 2. Exploratory Data Analysis (EDA) & Target Distribution

In [2]:
# Check missing values and summary statistics
print("Missing values per column:")
print(df.isnull().sum())

df.describe().T

In [3]:
# Target Class Balance & Risk Factors by Gender
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Target Distribution
sns.countplot(x='target', data=df, palette=['#34C759', '#FF3B30'], ax=axes[0])
axes[0].set_title('Target Distribution: Heart Disease (1) vs Normal (0)', fontsize=13, fontweight='bold')
axes[0].set_xticklabels(['No Disease', 'Heart Disease Present'])

# Heart Disease Frequency by Gender
sns.countplot(x='sex', hue='target', data=df, palette=['#007AFF', '#FF2D55'], ax=axes[1])
axes[1].set_title('Heart Disease Frequency by Biological Sex', fontsize=13, fontweight='bold')
axes[1].set_xticklabels(['Female', 'Male'])
axes[1].legend(['No Disease', 'Heart Disease'])

plt.tight_layout()
plt.show()

In [4]:
# Age vs Max Heart Rate (thalach) Stratified by Diagnosis
plt.figure(figsize=(10, 6))
sns.scatterplot(x='age', y='thalach', hue='target', palette=['#34C759', '#FF3B30'], data=df, alpha=0.85, s=60)
plt.title('Age vs. Maximum Heart Rate (Thalach) Stratified by Heart Disease', fontsize=14, fontweight='bold')
plt.xlabel('Age (Years)')
plt.ylabel('Max Heart Rate Achieved (bpm)')
plt.legend(['No Disease', 'Disease Present'])
plt.show()

## ⚙️ 3. Preprocessing, Standardization & Feature Pipeline

In [5]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve

# Features and Target
X = df.drop(columns=['target'])
y = df['target']

continuous_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
categorical_cols = [c for c in X.columns if c not in continuous_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), continuous_cols),
        ('cat', 'passthrough', categorical_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

## 🤖 4. Model Training, Benchmark Comparisons & Hyperparameter Optimization

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Support Vector Machine': SVC(probability=True, random_state=42)
}

results = []

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('clf', model)
    ])
    
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='roc_auc')
    pipe.fit(X_train, y_train)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, y_prob)
    
    results.append({
        'Model': name,
        'CV ROC-AUC (Mean)': np.mean(cv_scores),
        'CV ROC-AUC (Std)': np.std(cv_scores),
        'Test ROC-AUC': test_auc
    })

results_df = pd.DataFrame(results).sort_values(by='Test ROC-AUC', ascending=False)
results_df

## 📈 5. Fine-Tuning & Diagnostic ROC Curve Evaluation

In [7]:
# Hyperparameter optimization on best performing model (Random Forest)
rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', RandomForestClassifier(random_state=42))
])

param_grid = {
    'clf__n_estimators': [50, 100, 150],
    'clf__max_depth': [3, 5, 8],
    'clf__min_samples_split': [2, 4]
}

grid = GridSearchCV(rf_pipe, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print(f"Best Parameters: {grid.best_params_}")
print(f"Test ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Heart Disease']))

In [8]:
# Plot Final Evaluation Metrics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
ax1.plot(fpr, tpr, color='#007AFF', lw=2.5, label=f'Optimized Model (AUC = {roc_auc_score(y_test, y_prob):.4f})')
ax1.plot([0, 1], [0, 1], color='gray', linestyle='--')
ax1.set_title('Receiver Operating Characteristic (ROC) Curve', fontsize=13, fontweight='bold')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.legend(loc='lower right')

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax2,
            xticklabels=['No Disease', 'Heart Disease'],
            yticklabels=['No Disease', 'Heart Disease'])
ax2.set_title('Diagnostic Confusion Matrix', fontsize=13, fontweight='bold')
ax2.set_xlabel('Predicted Diagnosis')
ax2.set_ylabel('Actual Clinical Outcome')

plt.tight_layout()
plt.show()

## 💾 6. Pipeline Export for Clinical Decision Web Deployment

In [9]:
import pickle

os.makedirs('../models', exist_ok=True)
with open('../models/heart_disease_pipeline.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print("✅ ML Pipeline successfully exported to ../models/heart_disease_pipeline.pkl!")